# Phase 1 — Ingestion + bare NL2SQL loop

Goal: load a CSV into SQLite, pull its schema, and get one working function
that turns a question into SQL and runs it.

No LangGraph, no RAG yet — just the core loop tested directly here.
Get this solid on your own dataset before moving to notebook 02.

In [ ]:

import pandas as pd
import sqlite3
from groq import Groq


#client  = Groq   # reads GROQ_API_KEY from env auto 
MODEL   = 'llama-3.3-70b-versatile'
DB_PATH = 'data.db'

In [3]:
def load_csv_to_sqlite(csv_path, db_path=DB_PATH, table_name='data'):
    df = pd.read_csv(csv_path)
    # clean column names: lowercase + underscores
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    conn = sqlite3.connect(db_path)
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    conn.close()
    print(f'Loaded {len(df)} rows into table: {table_name}')
    return table_name

# swap path below to your own CSV
table_name = load_csv_to_sqlite('../job_postings_01-2024.csv')

Loaded 52924 rows into table: data


## Step 2 — Extract schema

In [4]:
def get_schema(table_name, db_path=DB_PATH):
    conn   = sqlite3.connect(db_path)
    cols   = conn.execute(f'PRAGMA table_info({table_name})').fetchall()
    sample = conn.execute(f'SELECT * FROM {table_name} LIMIT 3').fetchall()
    conn.close()
    col_lines    = [f'  - {c[1]} ({c[2]})' for c in cols]
    sample_lines = [str(row) for row in sample]
    return (
        f'Table: {table_name}\nColumns:\n' + '\n'.join(col_lines)
        + '\nSample rows:\n' + '\n'.join(sample_lines)
    )

schema_text = get_schema(table_name)
print(schema_text)

Table: data
Columns:
  - job_title_short (TEXT)
  - job_title (TEXT)
  - job_location (TEXT)
  - job_via (TEXT)
  - job_schedule_type (TEXT)
  - job_work_from_home (INTEGER)
  - search_location (TEXT)
  - job_posted_date (TEXT)
  - job_no_degree_mention (INTEGER)
  - job_health_insurance (INTEGER)
  - job_country (TEXT)
  - salary_rate (TEXT)
  - salary_year_avg (REAL)
  - salary_hour_avg (REAL)
  - company_name (TEXT)
  - job_skills (TEXT)
  - job_type_skills (TEXT)
Sample rows:
('Data Analyst', 'Summer Internship -Data Analyst Intern, Risk Management', 'Marlborough, MA', 'via Boatingrevealed.com', 'Full-time, Part-time, and Internship', 0, 'New York, United States', '2024-01-01 00:00:01', 0, 1, 'United States', None, None, None, "BJ's Wholesale Club", "['excel']", "{'analyst_tools': ['excel']}")
('Data Analyst', 'Staff Data Analyst Operations, Infrastructure & Systems', 'Fremont, CA', 'via ClimateTechList', 'Full-time', 0, 'California, United States', '2024-01-01 00:00:11', 1, 0, 'Un

In [8]:
schema_text += """

Column usage hints:
- Use job_title_short for grouping/aggregating by role (e.g. 'Data Analyst', 'Data Engineer')
- Use job_title only when the full job title is specifically requested
- job_work_from_home is boolean: 1 = True, 0 = False
- salary_year_avg and salary_hour_avg have many nulls — use AVG() which ignores them
- job_skills contains comma-separated skills e.g. 'python, sql, tableau' — use LIKE for filtering
"""

## Step 3 — NL to SQL via Groq

In [10]:
def ask_question(question, schema, model=MODEL):
    prompt = (
        'You are a SQLite expert. Write ONE SELECT query that answers the question.\n'
        'Return ONLY raw SQL — no markdown fences, no explanation.\n\n'
        f'{schema}\n\nQuestion: {question}\nSQL:'
    )
    resp = client.chat.completions.create(
        model=model,
        max_tokens=300,
        messages=[{'role': 'user', 'content': prompt}],
    )
    sql = resp.choices[0].message.content.strip()
    # strip markdown fences if the model adds them
    return sql.strip('`').removeprefix('sql').strip()

In [11]:
def run_sql(sql, db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    df   = pd.read_sql_query(sql, conn)
    conn.close()
    return df

In [12]:
#Testing
questions = [
    # SQL path
    "What are the top 5 most common job titles?",
    "How many jobs offer work from home?",
    "What is the average yearly salary by job title?",
    "Which country has the most job postings?",
    
    # Pandas path (stats/profiling)
    "What is the distribution of salary_year_avg?",
    "How many rows and columns does this dataset have?",
    "Which columns have missing values and how many?",
    "What is the average hourly salary across all jobs?",
    
    # test the skills LIKE query
"What are the top 5 jobs that require Python skills?",

# test null handling
"How many jobs have salary information available?",
]

for q in questions:
    print(f'Q: {q}')
    sql = ask_question(q, schema_text)
    print(f'SQL: {sql}')
    try:
        print(run_sql(sql))
    except Exception as e:
        print(f'ERROR: {e}')
    print('-' * 50)

Q: What are the top 5 most common job titles?
SQL: SELECT job_title_short, COUNT(*) as count FROM data GROUP BY job_title_short ORDER BY count DESC LIMIT 5
        job_title_short  count
0         Data Engineer  14060
1          Data Analyst  12328
2        Data Scientist  11422
3  Senior Data Engineer   3161
4      Business Analyst   3083
--------------------------------------------------
Q: How many jobs offer work from home?
SQL: SELECT COUNT(*) FROM data WHERE job_work_from_home = 1
   COUNT(*)
0      7319
--------------------------------------------------
Q: What is the average yearly salary by job title?
SQL: SELECT job_title_short, AVG(salary_year_avg) AS average_yearly_salary FROM data GROUP BY job_title_short
             job_title_short  average_yearly_salary
0           Business Analyst           86972.991071
1             Cloud Engineer          118950.000000
2               Data Analyst           84261.890053
3              Data Engineer          125276.328971
4           